# Visual Contrastive Decoding (VCD) - POPE Benchmark on Kaggle

This notebook runs the **POPE (Polling-based Object Probing Evaluation)** benchmark across 3 splits (`random`, `popular`, `adversarial`) using **Visual Contrastive Decoding (VCD)** on Kaggle with free **GPU 2xT4**.

### Key Evaluation Constraints:
- **GPU**: Kaggle 2xT4 (32GB VRAM total)
- **Precision**: Full BF16 (`torch.bfloat16`, NO quantization) for maximum benchmark fidelity
- **Generation**: `max_new_tokens = 6`, Greedy Decoding (`do_sample = False`, `temperature = 0.0`)
- **Models supported**: `llava` (LLaVA-1.5-7B) or `qwen2vl` (Qwen2-VL-7B-Instruct)
- **Prompt format**: Qwen2-VL automatically appends `"Please answer with yes or no."`
- **Method**: Visual Contrastive Decoding (`use_vcd=True`, `noise_step=500`, `alpha=1.0`, `beta=0.1`)

### Prerequisites Before Running:
1. **Accelerator**: Select **GPU T4 x2** (Notebook Settings -> Accelerator).
2. **Internet**: Toggle **Internet ON** (Notebook Settings -> Internet).
3. **HuggingFace Token**: Add a secret named `HF_TOKEN` in **Add-ons -> Secrets**.
4. **COCO 2014 val images**: Add/mount dataset in **+ Add Input** (e.g. `val2014` or `datasets/biminhco/val2014/val2014`).


In [ ]:
# Cell 1: Environment Setup & Dependencies Installation
# 1. Gỡ bỏ torchaudio để giải quyết dứt điểm xung đột CUDA version mismatch
!pip uninstall -y -q torchaudio

# 2. Cài đặt các thư viện cần thiết (không cài torchvision/torch để tránh lệch CUDA Kaggle)
!pip install -q --no-cache-dir \
    "transformers>=4.45.0" \
    "accelerate>=0.26.0" \
    sentencepiece \
    protobuf \
    tiktoken \
    qwen_vl_utils \
    pyyaml \
    tqdm \
    huggingface_hub \
    pandas

print("✅ Dependencies successfully installed!")
from transformers import AutoProcessor
print("✅ AutoProcessor import verified successfully!")


In [ ]:
# 2. Authenticate with Hugging Face via Kaggle Secrets
import os
from kaggle_secrets import UserSecretsClient
import huggingface_hub

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    huggingface_hub.login(token=hf_token)
    print(" Successfully logged in to Hugging Face Hub!")
except Exception as e:
    print(f"[Notice] Kaggle Secrets login failed: {e}")
    print("If accessing gated models, please ensure 'HF_TOKEN' is added in Add-ons -> Secrets.")


In [ ]:
# 3. Clone Repository or Navigate to Codebase
import os

# Path where the repo is cloned or mounted
repo_path = "/kaggle/working/VCD"

if not os.path.exists(repo_path):
    # Clone your VCD repository containing vcd_experiments
    !git clone https://github.com/ntmy12/VCD.git {repo_path}
else:
    print(f" Repository already present at {repo_path}")

# Enter vcd_experiments directory
%cd /kaggle/working/VCD/vcd_experiments


In [ ]:
# 4. Environment & Hardware Verification
import torch
import yaml
import os

print("=== GPU Environment ===")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        prop = torch.cuda.get_device_properties(i)
        print(f"  Device {i}: {prop.name} ({prop.total_memory / 1e9:.2f} GB VRAM)")

print("\n=== Dataset Path Verification ===")
config_file = "configs/data_paths_kaggle.yaml"
with open(config_file, "r") as f:
    cfg = yaml.safe_load(f)

coco_img_dir = cfg.get("coco_val2014_images", "")
pope_anno_dir = cfg.get("pope_coco_annotation_dir", "")

# Auto-discovery fallback if configured directory is not immediately present
if not os.path.isdir(coco_img_dir):
    print(f"Configured path not found: {coco_img_dir}. Auto-detecting in /kaggle/input/...")
    if os.path.isdir("/kaggle/input"):
        for root, dirs, files in os.walk("/kaggle/input"):
            if any(f.startswith("COCO_val2014_") for f in files[:10]):
                coco_img_dir = root
                print(f" Found image directory at: {coco_img_dir}")
                break

print(f"Using Image Dir: {coco_img_dir}")
if os.path.isdir(coco_img_dir):
    imgs = os.listdir(coco_img_dir)[:3]
    print(f"   Image directory verified! Samples: {imgs}")
else:
    print(f"   Image directory NOT found. Check contents of /kaggle/input/:")
    !ls -la /kaggle/input/

print(f"\nConfigured POPE Annotation Dir: {pope_anno_dir}")
if os.path.isdir(pope_anno_dir):
    print(f"   Annotation directory exists! Files: {os.listdir(pope_anno_dir)}")
else:
    print(f"   Annotation dir not found at {pope_anno_dir}. (run_pope.py will fallback to internal repo data)")


In [ ]:
# 5. Run POPE Evaluation (Random, Popular, Adversarial)
# Choose model: 'llava' or 'qwen2vl'
MODEL = "llava"

# Execution parameters:
# --split all: runs all 3 splits sequentially using the same loaded model
# --use_vcd: applies Visual Contrastive Decoding (noise_step=500, alpha=1.0, beta=0.1)
# max_new_tokens=6 and greedy decoding are hardcoded in run_pope.py
!python benchmarks/pope/run_pope.py \
    --model {MODEL} \
    --split all \
    --use_vcd \
    --config_path configs/data_paths_kaggle.yaml \
    --noise_step 500 \
    --cd_alpha 1.0 \
    --cd_beta 0.1 \
    --seed 42


In [ ]:
# 6. Display POPE Benchmark Summary & Metrics
import glob
import json
import pandas as pd

run_dirs = sorted(glob.glob("results/*pope*"))
if run_dirs:
    latest_dir = run_dirs[-1]
    summary_file = os.path.join(latest_dir, "summary_all_splits.json")
    if os.path.exists(summary_file):
        with open(summary_file, "r") as f:
            summary_data = json.load(f)
        df = pd.DataFrame(summary_data).T[["Accuracy", "Precision", "Recall", "F1", "Yes_ratio", "Total", "Unknown"]]
        print(f"\n{'='*30} BENCHMARK RESULTS ({MODEL.upper()} + VCD) {'='*30}")
        display(df)
    else:
        print(f"Checking split directories in {latest_dir}...")
        for m_file in glob.glob(f"{latest_dir}/**/metrics.json", recursive=True):
            split_name = os.path.basename(os.path.dirname(m_file))
            with open(m_file, "r") as f:
                print(f"\n--- {split_name.upper()} ---")
                print(json.dumps(json.load(f), indent=2))
else:
    print("No results found in results/ directory yet.")
